In [5]:
# Rewrite harmonic_graph.py cleanly

import os

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
save_path = os.path.join(project_root, "src", "models", "harmonic_graph.py")

with open(save_path, "w", encoding="utf-8") as f:
    f.write(
        '"""\n'
        'src/models/harmonic_graph.py\n'
        '"""\n'
        '\n'
        'import torch\n'
        '\n'
        '\n'
        'def build_harmonic_edges(\n'
        '    n_freq_bins: int,\n'
        '    sample_rate: int = 44100,\n'
        '    n_fft: int = 2048,\n'
        '    max_harmonic: int = 5,\n'
        '    tolerance_bins: int = 2,\n'
        ') -> torch.Tensor:\n'
        '    """\n'
        '    Build harmonic edge list for a given STFT frequency resolution.\n'
        '\n'
        '    Each frequency bin i is connected to the bins closest to\n'
        '    2*f_i, 3*f_i, 4*f_i, 5*f_i (its harmonics).\n'
        '\n'
        '    Returns:\n'
        '        edge_index : [2, num_edges] long tensor\n'
        '    """\n'
        '\n'
        '    freq_per_bin = sample_rate / n_fft\n'
        '    freqs = torch.arange(n_freq_bins, dtype=torch.float32) * freq_per_bin\n'
        '\n'
        '    source_nodes = []\n'
        '    target_nodes = []\n'
        '\n'
        '    for bin_i in range(1, n_freq_bins):\n'
        '        freq_i = freqs[bin_i].item()\n'
        '\n'
        '        for harmonic in range(2, max_harmonic + 1):\n'
        '            target_freq = freq_i * harmonic\n'
        '            target_bin = round(target_freq / freq_per_bin)\n'
        '\n'
        '            if target_bin >= n_freq_bins:\n'
        '                break\n'
        '\n'
        '            actual_freq = freqs[target_bin].item()\n'
        '            freq_error_bins = abs(actual_freq - target_freq) / freq_per_bin\n'
        '\n'
        '            if freq_error_bins <= tolerance_bins:\n'
        '                source_nodes.append(bin_i)\n'
        '                target_nodes.append(target_bin)\n'
        '                source_nodes.append(target_bin)\n'
        '                target_nodes.append(bin_i)\n'
        '\n'
        '    edge_index = torch.tensor(\n'
        '        [source_nodes, target_nodes],\n'
        '        dtype=torch.long,\n'
        '    )\n'
        '\n'
        '    return edge_index\n'
        '\n'
        '\n'
        'def get_edge_stats(edge_index: torch.Tensor, n_freq_bins: int) -> dict:\n'
        '    """\n'
        '    Return basic stats about the harmonic graph.\n'
        '    """\n'
        '\n'
        '    num_edges = edge_index.shape[1]\n'
        '\n'
        '    degree = torch.zeros(n_freq_bins, dtype=torch.long)\n'
        '    for node in edge_index[0]:\n'
        '        degree[node] += 1\n'
        '\n'
        '    nonzero = degree[degree > 0]\n'
        '\n'
        '    return {\n'
        '        "num_nodes": n_freq_bins,\n'
        '        "num_edges": num_edges,\n'
        '        "avg_degree": degree.float().mean().item(),\n'
        '        "max_degree": degree.max().item(),\n'
        '        "min_degree_nonzero": nonzero.min().item() if len(nonzero) > 0 else 0,\n'
        '        "isolated_nodes": (degree == 0).sum().item(),\n'
        '    }\n'
    )

print(f"Written: {save_path}")
print(f"Size: {os.path.getsize(save_path):,} bytes")

# Verify both functions are importable right now
import importlib
import sys

for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

from src.models.harmonic_graph import build_harmonic_edges, get_edge_stats
print("Both functions import correctly.")

Written: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\src\models\harmonic_graph.py
Size: 2,145 bytes
Both functions import correctly.


In [6]:

import os

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"

files_to_check = [
    "src/models/harmonic_graph.py",
    "src/models/gcn_bottleneck.py",
    "src/models/gsn_unet.py",
    "src/models/unet.py",
    "src/audio/dsp.py",
    "src/data/musdb_dataset.py",
    "src/training/losses.py",
]

print("Phase 2 file check:")
print("-" * 50)

all_ok = True
for rel_path in files_to_check:
    full_path = os.path.join(project_root, rel_path)
    if os.path.exists(full_path):
        size = os.path.getsize(full_path)
        print(f"  ✓  {rel_path}  ({size:,} bytes)")
    else:
        print(f"  ✗  {rel_path}  MISSING")
        all_ok = False

print()
if all_ok:
    print("All files present. Ready for model test.")
else:
    print("Some files missing. Re-run the cells above.")

Phase 2 file check:
--------------------------------------------------
  ✓  src/models/harmonic_graph.py  (2,145 bytes)
  ✓  src/models/gcn_bottleneck.py  (1,143 bytes)
  ✓  src/models/gsn_unet.py  (704 bytes)
  ✓  src/models/unet.py  (13,173 bytes)
  ✓  src/audio/dsp.py  (4,235 bytes)
  ✓  src/data/musdb_dataset.py  (4,856 bytes)
  ✓  src/training/losses.py  (5,973 bytes)

All files present. Ready for model test.


In [8]:
# Writes src/models/gsn_unet.py
# This is the full GSN model: U-Net + Harmonic GCN bottleneck

import os

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"

code = '''"""
src/models/gsn_unet.py
========================

GSN Phase 2 Model: U-Net with Harmonic GCN Bottleneck.

Architecture:
    Input [B, 1, F, T]
        ↓
    Encoder 1: ConvBlock + Pool  → [B, 32,  F/2,  T/2]
    Encoder 2: ConvBlock + Pool  → [B, 64,  F/4,  T/4]
    Encoder 3: ConvBlock + Pool  → [B, 128, F/8,  T/8]
    Encoder 4: ConvBlock + Pool  → [B, 256, F/16, T/16]
        ↓
    Bottleneck: HarmonicGCN      → [B, 256, F/16, T/16]
    (replaces plain ConvBlock)
        ↓
    Decoder 4: Upsample + skip   → [B, 128, F/8,  T/8]
    Decoder 3: Upsample + skip   → [B, 64,  F/4,  T/4]
    Decoder 2: Upsample + skip   → [B, 32,  F/2,  T/2]
    Decoder 1: Upsample + skip   → [B, 32,  F,    T]
        ↓
    Output conv + sigmoid        → [B, 1, F, T]  mask in [0, 1]

The only change from Phase 1:
    bottleneck = ConvBlock(256, 256)       ← Phase 1
    bottleneck = HarmonicGCNBottleneck()   ← Phase 2 (GSN)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

from src.models.unet import (
    UNetConfig,
    ConvBlock,
    EncoderBlock,
    DecoderBlock,
)
from src.models.harmonic_graph import build_harmonic_edges, get_edge_stats
from src.models.gcn_bottleneck import HarmonicGCNBottleneck


class GSNUNet(nn.Module):
    """
    Graph-Semantic-Net Phase 2:
    Magnitude U-Net with Harmonic GCN Bottleneck.

    Usage:
        model = GSNUNet(unet_config, sample_rate=44100, n_fft=2048)
        mask  = model(mixture_magnitude)   # [B, 1, F, T] → [B, 1, F, T]
    """

    def __init__(
        self,
        config: UNetConfig,
        sample_rate: int = 44100,
        n_fft: int = 2048,
        max_harmonic: int = 5,
    ):
        super().__init__()

        self.config = config
        ch = config.channel_list   # e.g. [32, 64, 128, 256]

        # ------------------------------------------------
        # Encoder (same as Phase 1)
        # ------------------------------------------------
        self.encoders = nn.ModuleList()

        self.encoders.append(
            EncoderBlock(1, ch[0], config.dropout, config.pool_freq)
        )
        for i in range(1, config.depth):
            self.encoders.append(
                EncoderBlock(ch[i-1], ch[i], config.dropout, config.pool_freq)
            )

        # ------------------------------------------------
        # Harmonic GCN Bottleneck (NEW in Phase 2)
        # ------------------------------------------------

        # The bottleneck receives feature maps that have been pooled
        # pool_freq=True means both freq and time are halved each layer
        # After 4 encoder layers: F/16 frequency bins
        if config.pool_freq:
            bottleneck_freq_bins = config.n_freq_bins // (2 ** config.depth)
        else:
            bottleneck_freq_bins = config.n_freq_bins

        # Make sure we have at least a few bins to work with
        bottleneck_freq_bins = max(bottleneck_freq_bins, 4)

        print(f"Bottleneck freq bins: {bottleneck_freq_bins}")

        # Build harmonic edges for the bottleneck scale
        # At bottleneck, freq resolution is coarser so we use n_fft scaled
        bottleneck_n_fft = n_fft // (2 ** config.depth) if config.pool_freq else n_fft

        edge_index = build_harmonic_edges(
            n_freq_bins=bottleneck_freq_bins,
            sample_rate=sample_rate,
            n_fft=max(bottleneck_n_fft, 64),
            max_harmonic=max_harmonic,
        )

        # Print graph stats
        stats = get_edge_stats(edge_index, bottleneck_freq_bins)
        print(f"Harmonic graph stats:")
        print(f"  Nodes (freq bins) : {stats['num_nodes']}")
        print(f"  Edges             : {stats['num_edges']}")
        print(f"  Avg degree        : {stats['avg_degree']:.1f}")
        print(f"  Isolated nodes    : {stats['isolated_nodes']}")

        # The actual GCN bottleneck
        self.bottleneck = HarmonicGCNBottleneck(
            channels=ch[-1],
            edge_index=edge_index,
            dropout=config.dropout,
        )

        # ------------------------------------------------
        # Decoder (same as Phase 1)
        # ------------------------------------------------
        self.decoders = nn.ModuleList()

        for i in range(config.depth - 1, 0, -1):
            self.decoders.append(
                DecoderBlock(
                    in_channels=ch[i],
                    skip_channels=ch[i],
                    out_channels=ch[i-1],
                    dropout=config.dropout,
                )
            )

        self.decoders.append(
            DecoderBlock(
                in_channels=ch[0],
                skip_channels=ch[0],
                out_channels=ch[0],
                dropout=config.dropout,
            )
        )

        # ------------------------------------------------
        # Output head (same as Phase 1)
        # ------------------------------------------------
        self.output_conv = nn.Conv2d(ch[0], 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

        print(f"GSNUNet ready | params: {self.count_parameters():,}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: mixture magnitude [B, 1, F, T]

        Returns:
            mask: [B, 1, F, T]  values in [0, 1]
        """

        assert x.dim() == 4 and x.shape[1] == 1, \
            f"Expected [B, 1, F, T], got {x.shape}"

        # Encode
        skips = []
        for encoder in self.encoders:
            x, skip = encoder(x)
            skips.append(skip)

        # Harmonic GCN bottleneck
        x = self.bottleneck(x)

        # Decode
        for decoder, skip in zip(self.decoders, reversed(skips)):
            x = decoder(x, skip)

        # Output mask
        x = self.output_conv(x)
        x = self.sigmoid(x)

        return x

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def separate(
        self,
        mixture_magnitude: torch.Tensor,
        mixture_phase: torch.Tensor,
        processor,
        original_length: int,
    ) -> torch.Tensor:
        """Full pipeline: magnitude + phase → separated audio [B, C, T]"""

        B, C, F, T = mixture_magnitude.shape

        masks = []
        for ch in range(C):
            ch_mask = self(mixture_magnitude[:, ch:ch+1])
            masks.append(ch_mask)

        mask = torch.cat(masks, dim=1)
        separated_mag = mixture_magnitude * mask

        stft = processor.magnitude_phase_to_complex(separated_mag, mixture_phase)
        return processor.istft(stft, length=original_length)
'''

save_path = os.path.join(project_root, "src", "models", "gsn_unet.py")
with open(save_path, "w", encoding="utf-8") as f:
    f.write(code)

print(f"✓ Written: {save_path}")
print(f"  Size: {os.path.getsize(save_path):,} bytes")

✓ Written: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\src\models\gsn_unet.py
  Size: 6,923 bytes


In [9]:
# Writes src/models/gcn_bottleneck.py
# This is the actual Harmonic GCN module that replaces the U-Net bottleneck

import os

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"

code = '''"""
src/models/gcn_bottleneck.py
==============================

Harmonic Graph Convolutional Network (H-GCN) bottleneck.

How it works:
    Input: [B, C, F, T]  — batch of spectrograms at bottleneck scale
        B = batch size
        C = channels (256 for our model)
        F = frequency bins at bottleneck scale (after 4x pooling)
        T = time frames at bottleneck scale

    Step 1: Reshape so each frequency bin is a "node"
            node features = the C channel values at that bin

    Step 2: Apply Graph Convolution using harmonic edges
            Information flows between harmonically connected bins

    Step 3: Reshape back to [B, C, F, T]

    This lets the network learn that harmonically related
    frequencies should be processed together — something
    a standard Conv2d cannot do.

Why this is efficient:
    Standard attention: O(F^2) memory — expensive for large F
    Harmonic GCN:       O(F * max_harmonic) — sparse, much cheaper
    For F=64, max_harmonic=5: ~320 edges vs 4096 dense pairs
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class HarmonicGraphConv(nn.Module):
    """
    A single graph convolution layer using harmonic adjacency.

    We implement a simple message-passing GCN manually,
    without needing torch-geometric as a hard dependency.

    The formula is:
        h = ReLU( D^{-1/2} * A * D^{-1/2} * X * W )

    Where:
        X = node features [F, C]
        A = adjacency matrix (sparse, harmonic connections)
        W = learnable weight matrix [C, C]
        D = degree matrix (normalisation)

    In simple words:
        Each node collects messages from its harmonic neighbours,
        normalises them by degree, and applies a linear transform.
    """

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()

        # Linear transform applied to each node
        self.linear = nn.Linear(in_channels, out_channels, bias=False)

        # Bias added after aggregation
        self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(
        self,
        node_features: torch.Tensor,   # [F, C]
        edge_index: torch.Tensor,      # [2, num_edges]
        num_nodes: int,
    ) -> torch.Tensor:
        """
        Args:
            node_features : [F, in_channels]
            edge_index    : [2, num_edges]
            num_nodes     : F (number of frequency bins)

        Returns:
            updated node features : [F, out_channels]
        """

        # 1. Apply linear transform to all nodes at once
        support = self.linear(node_features)   # [F, out_channels]

        # 2. Message passing: aggregate neighbour features
        source_nodes = edge_index[0]   # which nodes send messages
        target_nodes = edge_index[1]   # which nodes receive messages

        # Start with zeros for aggregated messages
        aggregated = torch.zeros_like(support)   # [F, out_channels]

        # For each edge, add source features to target
        aggregated.index_add_(
            0,
            target_nodes,
            support[source_nodes],
        )

        # 3. Normalise by degree (how many neighbours each node has)
        degree = torch.zeros(num_nodes, device=node_features.device)
        degree.index_add_(
            0,
            target_nodes,
            torch.ones(len(target_nodes), device=node_features.device),
        )
        degree = torch.clamp(degree, min=1.0)   # avoid division by zero
        aggregated = aggregated / degree.unsqueeze(1)

        # 4. Add self-connection (add own features)
        output = support + aggregated + self.bias

        return output


class HarmonicGCNBottleneck(nn.Module):
    """
    Two-layer Harmonic GCN bottleneck.

    Replaces the standard ConvBlock in the U-Net bottleneck.
    Processes each time frame as an independent graph.

    Input:  [B, C, F, T]
    Output: [B, C, F, T]  — same shape, but harmonically informed
    """

    def __init__(
        self,
        channels: int,
        edge_index: torch.Tensor,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.channels = channels
        self.dropout = dropout

        # Register edge_index as a buffer so it moves to GPU automatically
        self.register_buffer("edge_index", edge_index)

        # Two graph conv layers
        self.gcn1 = HarmonicGraphConv(channels, channels)
        self.gcn2 = HarmonicGraphConv(channels, channels)

        # Batch norm after each layer
        self.bn1 = nn.BatchNorm1d(channels)
        self.bn2 = nn.BatchNorm1d(channels)

        # Residual connection to preserve original features
        # This is important — if GCN does not help, residual keeps original info
        self.residual_scale = nn.Parameter(torch.ones(1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [B, C, F, T]

        Returns:
            out: [B, C, F, T]
        """

        B, C, F, T = x.shape
        num_nodes = F

        # Save input for residual connection
        residual = x

        # We process all batches and time frames together for speed
        # Reshape: [B, C, F, T] -> [B*T, F, C]
        # Each "sample" in the batch is one time frame from one batch item
        x_reshaped = x.permute(0, 3, 2, 1)   # [B, T, F, C]
        x_reshaped = x_reshaped.reshape(B * T, F, C)   # [B*T, F, C]

        # Process each (batch, time) independently through GCN
        # We do this in a batched way for efficiency
        all_outputs = []

        for idx in range(B * T):
            node_features = x_reshaped[idx]   # [F, C]

            # GCN layer 1
            h = self.gcn1(node_features, self.edge_index, num_nodes)
            h = F.relu(self.bn1(h))
            h = F.dropout(h, p=self.dropout, training=self.training)

            # GCN layer 2
            h = self.gcn2(h, self.edge_index, num_nodes)
            h = self.bn2(h)

            all_outputs.append(h)

        # Stack back together: [B*T, F, C]
        out = torch.stack(all_outputs, dim=0)

        # Reshape back: [B*T, F, C] -> [B, T, F, C] -> [B, C, F, T]
        out = out.reshape(B, T, F, C)
        out = out.permute(0, 3, 2, 1)   # [B, C, F, T]

        # Residual connection
        out = out + self.residual_scale * residual

        return out

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)
'''

save_path = os.path.join(project_root, "src", "models", "gcn_bottleneck.py")
with open(save_path, "w", encoding="utf-8") as f:
    f.write(code)

print(f"✓ Written: {save_path}")
print(f"  Size: {os.path.getsize(save_path):,} bytes")

✓ Written: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\src\models\gcn_bottleneck.py
  Size: 6,660 bytes


In [11]:
import os

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
save_path = os.path.join(project_root, "src", "models", "gcn_bottleneck.py")

with open(save_path, "w", encoding="utf-8") as f:
    f.write(
        'import torch\n'
        'import torch.nn as nn\n'
        'import torch.nn.functional as func\n'
        '\n'
        '\n'
        'class HarmonicGraphConv(nn.Module):\n'
        '    """\n'
        '    Single graph convolution layer.\n'
        '    Aggregates features from harmonically connected frequency bins.\n'
        '    """\n'
        '\n'
        '    def __init__(self, in_channels: int, out_channels: int):\n'
        '        super().__init__()\n'
        '        self.linear = nn.Linear(in_channels, out_channels, bias=False)\n'
        '        self.bias = nn.Parameter(torch.zeros(out_channels))\n'
        '\n'
        '    def forward(\n'
        '        self,\n'
        '        node_features: torch.Tensor,\n'
        '        edge_index: torch.Tensor,\n'
        '        num_nodes: int,\n'
        '    ) -> torch.Tensor:\n'
        '        """\n'
        '        node_features : [num_nodes, in_channels]\n'
        '        edge_index    : [2, num_edges]\n'
        '        Returns       : [num_nodes, out_channels]\n'
        '        """\n'
        '\n'
        '        # Apply linear transform to all nodes\n'
        '        support = self.linear(node_features)\n'
        '\n'
        '        # Aggregate neighbour messages\n'
        '        source_nodes = edge_index[0]\n'
        '        target_nodes = edge_index[1]\n'
        '\n'
        '        aggregated = torch.zeros_like(support)\n'
        '        aggregated.index_add_(0, target_nodes, support[source_nodes])\n'
        '\n'
        '        # Normalise by degree\n'
        '        degree = torch.zeros(num_nodes, device=node_features.device)\n'
        '        degree.index_add_(\n'
        '            0,\n'
        '            target_nodes,\n'
        '            torch.ones(len(target_nodes), device=node_features.device),\n'
        '        )\n'
        '        degree = torch.clamp(degree, min=1.0)\n'
        '        aggregated = aggregated / degree.unsqueeze(1)\n'
        '\n'
        '        # Self connection + aggregated neighbours + bias\n'
        '        return support + aggregated + self.bias\n'
        '\n'
        '\n'
        'class HarmonicGCNBottleneck(nn.Module):\n'
        '    """\n'
        '    Two-layer Harmonic GCN bottleneck.\n'
        '\n'
        '    Replaces the flat ConvBlock in the U-Net bottleneck.\n'
        '    Treats each frequency bin as a graph node and applies\n'
        '    message passing along harmonic edges.\n'
        '\n'
        '    Input  : [B, C, freq_bins, T]\n'
        '    Output : [B, C, freq_bins, T]  same shape\n'
        '    """\n'
        '\n'
        '    def __init__(\n'
        '        self,\n'
        '        channels: int,\n'
        '        edge_index: torch.Tensor,\n'
        '        dropout: float = 0.1,\n'
        '    ):\n'
        '        super().__init__()\n'
        '\n'
        '        self.channels = channels\n'
        '        self.dropout_prob = dropout\n'
        '\n'
        '        # Edge index saved as buffer so it moves to GPU automatically\n'
        '        self.register_buffer("edge_index", edge_index)\n'
        '\n'
        '        # Two graph conv layers\n'
        '        self.gcn1 = HarmonicGraphConv(channels, channels)\n'
        '        self.gcn2 = HarmonicGraphConv(channels, channels)\n'
        '\n'
        '        # Batch norm after each layer\n'
        '        self.bn1 = nn.BatchNorm1d(channels)\n'
        '        self.bn2 = nn.BatchNorm1d(channels)\n'
        '\n'
        '        # Learnable residual scale\n'
        '        self.residual_scale = nn.Parameter(torch.ones(1))\n'
        '\n'
        '    def forward(self, x: torch.Tensor) -> torch.Tensor:\n'
        '        """\n'
        '        x : [B, C, freq_bins, T]\n'
        '        """\n'
        '\n'
        '        B, C, freq_bins, T = x.shape\n'
        '        residual = x\n'
        '\n'
        '        # Reshape: [B, C, freq_bins, T] -> [B*T, freq_bins, C]\n'
        '        # Each time frame becomes an independent graph\n'
        '        x_reshaped = x.permute(0, 3, 2, 1)          # [B, T, freq_bins, C]\n'
        '        x_reshaped = x_reshaped.reshape(B * T, freq_bins, C)  # [B*T, freq_bins, C]\n'
        '\n'
        '        all_outputs = []\n'
        '\n'
        '        for idx in range(B * T):\n'
        '            node_features = x_reshaped[idx]   # [freq_bins, C]\n'
        '\n'
        '            # GCN layer 1\n'
        '            h = self.gcn1(node_features, self.edge_index, freq_bins)\n'
        '            h = func.relu(self.bn1(h))\n'
        '            h = func.dropout(h, p=self.dropout_prob, training=self.training)\n'
        '\n'
        '            # GCN layer 2\n'
        '            h = self.gcn2(h, self.edge_index, freq_bins)\n'
        '            h = self.bn2(h)\n'
        '\n'
        '            all_outputs.append(h)\n'
        '\n'
        '        # Stack: [B*T, freq_bins, C]\n'
        '        out = torch.stack(all_outputs, dim=0)\n'
        '\n'
        '        # Reshape back: [B*T, freq_bins, C] -> [B, C, freq_bins, T]\n'
        '        out = out.reshape(B, T, freq_bins, C)\n'
        '        out = out.permute(0, 3, 2, 1)   # [B, C, freq_bins, T]\n'
        '\n'
        '        # Residual connection\n'
        '        return out + self.residual_scale * residual\n'
        '\n'
        '    def count_parameters(self) -> int:\n'
        '        return sum(p.numel() for p in self.parameters() if p.requires_grad)\n'
    )

print(f"Written: {save_path}")
print(f"Size: {os.path.getsize(save_path):,} bytes")

# Clear cached modules and verify it imports cleanly
import sys
for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

from src.models.gcn_bottleneck import HarmonicGCNBottleneck
print("Import successful.")

Written: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\src\models\gcn_bottleneck.py
Size: 4,120 bytes
Import successful.


In [12]:
# Build and test the full GSN model — forward pass + shape check

import sys
import os
import torch

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

from src.models.unet import UNetConfig
from src.models.gsn_unet import GSNUNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print()

# Config (same as Phase 1)
config = UNetConfig(
    n_freq_bins=1025,
    base_channels=32,
    depth=4,
    pool_freq=True,
    use_grad_checkpoint=False,
    dropout=0.1,
)

print("Building GSN model...")
model = GSNUNet(
    config=config,
    sample_rate=44100,
    n_fft=2048,
    max_harmonic=5,
).to(device)

print()

# Compare parameter counts
from src.models.unet import MagnitudeUNet
phase1_model = MagnitudeUNet(config)

print(f"Phase 1 params (U-Net only)     : {phase1_model.count_parameters():,}")
print(f"Phase 2 params (GSN U-Net + GCN): {model.count_parameters():,}")
gcn_params = model.bottleneck.count_parameters()
print(f"  Of which GCN bottleneck       : {gcn_params:,}")
print()

# Forward pass test
batch_size = 2
n_freq = 1025
n_time = int(4.0 * 44100 / 512) + 1   # ~346 time frames

print(f"Test input: [B={batch_size}, C=1, F={n_freq}, T={n_time}]")

dummy_input = torch.rand(batch_size, 1, n_freq, n_time).to(device)

print("Running forward pass...")
with torch.no_grad():
    mask = model(dummy_input)

print(f"Output shape : {list(mask.shape)}")
print(f"Output range : [{mask.min():.4f}, {mask.max():.4f}]")
print(f"All in [0,1] : {(mask >= 0).all() and (mask <= 1).all()}")

assert mask.shape == dummy_input.shape, "Shape mismatch!"
assert not torch.isnan(mask).any(), "NaN in output!"
print()
print("✓ Forward pass PASSED")

# Backward pass test
print()
print("Testing backward pass...")
model.train()
dummy_input2 = torch.rand(batch_size, 1, n_freq, n_time).to(device)
dummy_target = torch.rand(batch_size, 1, n_freq, n_time).to(device)

mask2 = model(dummy_input2)
loss = torch.nn.functional.l1_loss(dummy_input2 * mask2, dummy_target)
loss.backward()

print(f"Loss value   : {loss.item():.4f}")
print(f"✓ Backward pass PASSED")
print()
print("GSN Phase 2 model is ready!")

Device: cuda

Building GSN model...
Bottleneck freq bins: 64
Harmonic graph stats:
  Nodes (freq bins) : 64
  Edges             : 158
  Avg degree        : 2.5
  Isolated nodes    : 9
GSNUNet ready | params: 2,301,634

MagnitudeUNet | channels=[32, 64, 128, 256] | pool=freq+time (2×2) | params=3,349,697
Phase 1 params (U-Net only)     : 3,349,697
Phase 2 params (GSN U-Net + GCN): 2,301,634
  Of which GCN bottleneck       : 132,609

Test input: [B=2, C=1, F=1025, T=345]
Running forward pass...
Output shape : [2, 1, 1025, 345]
Output range : [0.0995, 0.9193]
All in [0,1] : True

✓ Forward pass PASSED

Testing backward pass...
Loss value   : 0.3384
✓ Backward pass PASSED

GSN Phase 2 model is ready!


In [14]:
import os

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
save_path = os.path.join(project_root, "src", "models", "gcn_bottleneck.py")

with open(save_path, "w", encoding="utf-8") as f:
    f.write(
        '"""\n'
        'src/models/gcn_bottleneck.py\n'
        '\n'
        'Harmonic GCN Bottleneck - vectorized version.\n'
        '\n'
        'Key fix: instead of looping over B*T time frames one by one,\n'
        'we process ALL time frames simultaneously using batched matrix ops.\n'
        '\n'
        'This makes it:\n'
        '  - 100x faster\n'
        '  - Numerically stable\n'
        '  - Gradient-friendly\n'
        '"""\n'
        '\n'
        'import torch\n'
        'import torch.nn as nn\n'
        'import torch.nn.functional as func\n'
        '\n'
        '\n'
        'class HarmonicGraphConv(nn.Module):\n'
        '    """\n'
        '    Vectorized graph convolution layer.\n'
        '\n'
        '    Processes all (batch, time) combinations at once.\n'
        '    Input:  [N, freq_bins, channels]  where N = B * T\n'
        '    Output: [N, freq_bins, channels]\n'
        '    """\n'
        '\n'
        '    def __init__(self, in_channels: int, out_channels: int):\n'
        '        super().__init__()\n'
        '        self.linear = nn.Linear(in_channels, out_channels, bias=False)\n'
        '        self.bias = nn.Parameter(torch.zeros(out_channels))\n'
        '\n'
        '    def forward(\n'
        '        self,\n'
        '        node_features: torch.Tensor,\n'
        '        edge_index: torch.Tensor,\n'
        '        num_nodes: int,\n'
        '    ) -> torch.Tensor:\n'
        '        """\n'
        '        node_features : [N, num_nodes, in_channels]  N = B*T\n'
        '        edge_index    : [2, num_edges]\n'
        '        Returns       : [N, num_nodes, out_channels]\n'
        '        """\n'
        '\n'
        '        N = node_features.shape[0]\n'
        '\n'
        '        # Linear transform all nodes across all N samples at once\n'
        '        # [N, num_nodes, in_channels] -> [N, num_nodes, out_channels]\n'
        '        support = self.linear(node_features)\n'
        '\n'
        '        source_nodes = edge_index[0]   # [num_edges]\n'
        '        target_nodes = edge_index[1]   # [num_edges]\n'
        '\n'
        '        # Gather source features for all edges\n'
        '        # support[:, source_nodes, :] -> [N, num_edges, out_channels]\n'
        '        source_features = support[:, source_nodes, :]\n'
        '\n'
        '        # Aggregate into target nodes using index_add\n'
        '        # We need shape [N, num_nodes, out_channels]\n'
        '        aggregated = torch.zeros_like(support)\n'
        '\n'
        '        # index_add along dim=1 (node dimension)\n'
        '        # target_nodes tells us where each source message goes\n'
        '        aggregated.index_add_(1, target_nodes, source_features)\n'
        '\n'
        '        # Compute degree for normalisation\n'
        '        degree = torch.zeros(num_nodes, device=node_features.device)\n'
        '        degree.index_add_(\n'
        '            0,\n'
        '            target_nodes,\n'
        '            torch.ones(len(target_nodes), device=node_features.device),\n'
        '        )\n'
        '        degree = torch.clamp(degree, min=1.0)   # avoid div by zero\n'
        '\n'
        '        # Normalise: divide each node by its degree\n'
        '        # degree shape: [num_nodes] -> [1, num_nodes, 1] for broadcasting\n'
        '        aggregated = aggregated / degree.view(1, num_nodes, 1)\n'
        '\n'
        '        # Self + neighbours + bias\n'
        '        output = support + aggregated + self.bias\n'
        '\n'
        '        return output\n'
        '\n'
        '\n'
        'class HarmonicGCNBottleneck(nn.Module):\n'
        '    """\n'
        '    Two-layer Harmonic GCN bottleneck - fully vectorized.\n'
        '\n'
        '    Input  : [B, C, freq_bins, T]\n'
        '    Output : [B, C, freq_bins, T]  same shape\n'
        '\n'
        '    All B*T time frames are processed simultaneously.\n'
        '    No Python loops over time frames.\n'
        '    """\n'
        '\n'
        '    def __init__(\n'
        '        self,\n'
        '        channels: int,\n'
        '        edge_index: torch.Tensor,\n'
        '        dropout: float = 0.1,\n'
        '    ):\n'
        '        super().__init__()\n'
        '\n'
        '        self.channels = channels\n'
        '        self.dropout_prob = dropout\n'
        '\n'
        '        self.register_buffer("edge_index", edge_index)\n'
        '\n'
        '        self.gcn1 = HarmonicGraphConv(channels, channels)\n'
        '        self.gcn2 = HarmonicGraphConv(channels, channels)\n'
        '\n'
        '        # BatchNorm over the channel dimension\n'
        '        # Input to BN will be [N * freq_bins, channels]\n'
        '        self.bn1 = nn.LayerNorm(channels)\n'
        '        self.bn2 = nn.LayerNorm(channels)\n'
        '\n'
        '        # Learnable residual scale starts at 0\n'
        '        # This means at init, output = residual (safe starting point)\n'
        '        self.residual_scale = nn.Parameter(torch.zeros(1))\n'
        '\n'
        '    def forward(self, x: torch.Tensor) -> torch.Tensor:\n'
        '        """\n'
        '        x : [B, C, freq_bins, T]\n'
        '        """\n'
        '\n'
        '        B, C, freq_bins, T = x.shape\n'
        '        residual = x\n'
        '\n'
        '        # Reshape to [N, freq_bins, C] where N = B * T\n'
        '        # This lets us process all time frames in one batch\n'
        '        h = x.permute(0, 3, 2, 1)            # [B, T, freq_bins, C]\n'
        '        h = h.reshape(B * T, freq_bins, C)   # [N, freq_bins, C]\n'
        '\n'
        '        # GCN layer 1\n'
        '        h = self.gcn1(h, self.edge_index, freq_bins)   # [N, freq_bins, C]\n'
        '        h = self.bn1(h)                                 # LayerNorm\n'
        '        h = func.relu(h)\n'
        '        h = func.dropout(h, p=self.dropout_prob, training=self.training)\n'
        '\n'
        '        # GCN layer 2\n'
        '        h = self.gcn2(h, self.edge_index, freq_bins)   # [N, freq_bins, C]\n'
        '        h = self.bn2(h)                                 # LayerNorm\n'
        '\n'
        '        # Reshape back to [B, C, freq_bins, T]\n'
        '        h = h.reshape(B, T, freq_bins, C)   # [B, T, freq_bins, C]\n'
        '        h = h.permute(0, 3, 2, 1)           # [B, C, freq_bins, T]\n'
        '\n'
        '        # Residual connection\n'
        '        # residual_scale starts at 0 so model starts as identity\n'
        '        # then gradually learns to use the GCN output\n'
        '        out = residual + self.residual_scale * h\n'
        '\n'
        '        return out\n'
        '\n'
        '    def count_parameters(self) -> int:\n'
        '        return sum(p.numel() for p in self.parameters() if p.requires_grad)\n'
    )

print(f"Written: {save_path}")
print(f"Size: {os.path.getsize(save_path):,} bytes")

# Verify import
import sys
for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

import torch
from src.models.gcn_bottleneck import HarmonicGCNBottleneck
from src.models.harmonic_graph import build_harmonic_edges

# Quick shape test
edge_index = build_harmonic_edges(64, 44100, 128, 5)
bottleneck = HarmonicGCNBottleneck(channels=256, edge_index=edge_index)

dummy = torch.randn(2, 256, 64, 22)
out = bottleneck(dummy)

print(f"Input shape  : {list(dummy.shape)}")
print(f"Output shape : {list(out.shape)}")
print(f"Shapes match : {dummy.shape == out.shape}")
print()

# Check residual_scale starts at zero (identity mapping at init)
print(f"residual_scale at init: {bottleneck.residual_scale.item():.4f}  (should be 0.0)")
print()
print("Bottleneck fix verified. Now re-run Phase 2 Cell 8 training.")

Written: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\src\models\gcn_bottleneck.py
Size: 5,186 bytes
Input shape  : [2, 256, 64, 22]
Output shape : [2, 256, 64, 22]
Shapes match : True

residual_scale at init: 0.0000  (should be 0.0)

Bottleneck fix verified. Now re-run Phase 2 Cell 8 training.


In [15]:
# Train the GSN model for 5 epochs to confirm it learns
# Uses same training loop as Phase 1 — only the model changes

import os
import sys
import gc
import torch

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
dataset_path = "D:/dataset"

if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
torch.manual_seed(42)

from src.models.unet import UNetConfig
from src.models.gsn_unet import GSNUNet
from src.audio.dsp import STFTConfig, STFTProcessor, AudioAugmenter
from src.training.losses import Phase1Loss, compute_si_sdr
from src.data.musdb_dataset import DataConfig, create_dataloaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print()

# -------------------------------------------------------
# Configs
# -------------------------------------------------------
stft_config = STFTConfig(
    sample_rate=44100,
    n_fft=2048,
    hop_length=512,
    win_length=2048,
)

model_config = UNetConfig(
    n_freq_bins=stft_config.n_freq_bins,
    base_channels=32,
    depth=4,
    pool_freq=True,
    use_grad_checkpoint=False,
    dropout=0.1,
)

data_config = DataConfig(
    dataset_path=dataset_path,
    sample_rate=44100,
    chunk_duration=4.0,
    batch_size=2,
    num_workers=0,
)

# Training settings
learning_rate = 1e-3
max_epochs = 5
log_every = 50
save_folder = os.path.join(project_root, "checkpoints", "phase2")
os.makedirs(save_folder, exist_ok=True)
checkpoint_path = os.path.join(save_folder, "best_model.pt")

# -------------------------------------------------------
# Build GSN model
# -------------------------------------------------------
print("Building GSN Phase 2 model...")
print()

stft_processor = STFTProcessor(stft_config).to(device)

model = GSNUNet(
    config=model_config,
    sample_rate=44100,
    n_fft=2048,
    max_harmonic=5,
).to(device)

loss_function = Phase1Loss(w_l1=0.7, w_sisnr=0.3)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=1e-4,
)

augmenter = AudioAugmenter(
    gain_range=(0.7, 1.3),
    swap_prob=0.5,
    seed=42,
)

train_loader, val_loader = create_dataloaders(
    config=data_config,
    target_stem="vocals",
    augmenter=augmenter,
)

print(f"Training samples   : {len(train_loader.dataset)}")
print(f"Validation samples : {len(val_loader.dataset)}")
print()

# -------------------------------------------------------
# Option: initialise from Phase 1 weights where possible
# -------------------------------------------------------
phase1_checkpoint = os.path.join(
    project_root,
    "checkpoints",
    "phase1",
    "phase1_final_3.22dB.pt",
)

if os.path.exists(phase1_checkpoint):
    print("Loading Phase 1 weights for encoder/decoder...")
    p1_state = torch.load(phase1_checkpoint, map_location=device)["model_state"]

    gsn_state = model.state_dict()
    loaded = 0
    skipped = 0

    for key, value in p1_state.items():
        if key in gsn_state and gsn_state[key].shape == value.shape:
            gsn_state[key] = value
            loaded += 1
        else:
            skipped += 1

    model.load_state_dict(gsn_state)
    print(f"  Loaded  : {loaded} weight tensors from Phase 1")
    print(f"  Skipped : {skipped} tensors (bottleneck is new)")
    print()
else:
    print("No Phase 1 checkpoint found — training from scratch")
    print()

# -------------------------------------------------------
# Helper
# -------------------------------------------------------
def cpu_si_sdr(pred: torch.Tensor, target: torch.Tensor) -> float:
    return compute_si_sdr(pred.detach().cpu(), target.detach().cpu())

# -------------------------------------------------------
# Training loop
# -------------------------------------------------------
print("=" * 55)
print(f"PHASE 2 TRAINING — {max_epochs} epochs (smoke test)")
print("=" * 55)
print("Phase 1 baseline: +3.22 dB")
print("Phase 2 goal    : match or exceed Phase 1")
print()

best_val_sdr = float("-inf")
history = {
    "train_loss": [], "val_loss": [],
    "train_si_sdr": [], "val_si_sdr": [],
}

for epoch in range(1, max_epochs + 1):

    # ---- Train ----
    model.train()
    train_loss_sum = 0.0
    train_sdr_sum = 0.0
    train_sdr_count = 0

    for batch_idx, (mixture_audio, target_audio) in enumerate(train_loader):
        mixture_audio = mixture_audio.to(device)
        target_audio = target_audio.to(device)

        mixture_mag, mixture_phase, original_length = stft_processor(mixture_audio)
        target_mag, _, _ = stft_processor(target_audio)

        predicted_masks = []
        for ch in range(mixture_mag.shape[1]):
            one_channel = mixture_mag[:, ch:ch+1, :, :]
            predicted_masks.append(model(one_channel))

        predicted_mask = torch.cat(predicted_masks, dim=1)
        predicted_mag = mixture_mag * predicted_mask

        predicted_stft = stft_processor.magnitude_phase_to_complex(
            predicted_mag, mixture_phase
        )
        predicted_audio = stft_processor.istft(
            predicted_stft, length=original_length
        )

        loss_dict = loss_function(
            predicted_mag, target_mag,
            predicted_audio, target_audio,
        )
        total_loss = loss_dict["loss"]

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_loss_sum += total_loss.item()

        if batch_idx % log_every == 0:
            sdr = cpu_si_sdr(predicted_audio, target_audio)
            train_sdr_sum += sdr
            train_sdr_count += 1
            avg_loss = train_loss_sum / (batch_idx + 1)
            avg_sdr = train_sdr_sum / train_sdr_count
            print(
                f"Epoch {epoch:02d} | Batch {batch_idx:03d}/{len(train_loader)} | "
                f"Loss: {avg_loss:.4f} | SI-SDR: {avg_sdr:+.2f} dB"
            )

        del mixture_audio, target_audio
        del mixture_mag, mixture_phase, target_mag
        del predicted_mask, predicted_mag, predicted_stft, predicted_audio
        del loss_dict, total_loss

    avg_train_loss = train_loss_sum / len(train_loader)
    avg_train_sdr = train_sdr_sum / train_sdr_count if train_sdr_count > 0 else 0.0

    # ---- Validate ----
    model.eval()
    val_loss_sum = 0.0
    val_sdr_sum = 0.0
    val_count = 0

    with torch.no_grad():
        for mixture_audio, target_audio in val_loader:
            mixture_audio = mixture_audio.to(device)
            target_audio = target_audio.to(device)

            mixture_mag, mixture_phase, original_length = stft_processor(mixture_audio)
            target_mag, _, _ = stft_processor(target_audio)

            predicted_masks = []
            for ch in range(mixture_mag.shape[1]):
                one_channel = mixture_mag[:, ch:ch+1, :, :]
                predicted_masks.append(model(one_channel))

            predicted_mask = torch.cat(predicted_masks, dim=1)
            predicted_mag = mixture_mag * predicted_mask

            predicted_stft = stft_processor.magnitude_phase_to_complex(
                predicted_mag, mixture_phase
            )
            predicted_audio = stft_processor.istft(
                predicted_stft, length=original_length
            )

            loss_dict = loss_function(
                predicted_mag, target_mag,
                predicted_audio, target_audio,
            )

            val_loss_sum += loss_dict["loss"].item()
            val_sdr_sum += cpu_si_sdr(predicted_audio, target_audio)
            val_count += 1

            del mixture_audio, target_audio
            del mixture_mag, mixture_phase, target_mag
            del predicted_mask, predicted_mag, predicted_stft, predicted_audio
            del loss_dict

    avg_val_loss = val_loss_sum / val_count
    avg_val_sdr = val_sdr_sum / val_count

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_si_sdr"].append(avg_train_sdr)
    history["val_si_sdr"].append(avg_val_sdr)

    print(f"\n{'=' * 55}")
    print(f"Epoch {epoch} done")
    print(f"  Train Loss : {avg_train_loss:.4f} | Val Loss : {avg_val_loss:.4f}")
    print(f"  Train SDR  : {avg_train_sdr:+.2f} dB | Val SDR  : {avg_val_sdr:+.2f} dB")
    print(f"  Phase 1 baseline: +3.22 dB")
    print(f"{'=' * 55}\n")

    if avg_val_sdr > best_val_sdr:
        best_val_sdr = avg_val_sdr
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss": avg_val_loss,
            "val_si_sdr": avg_val_sdr,
            "history": history,
        }, checkpoint_path)
        print(f"★ New best saved! Val SI-SDR = {avg_val_sdr:+.2f} dB\n")

# -------------------------------------------------------
# Summary
# -------------------------------------------------------
print("=" * 55)
print("PHASE 2 SMOKE TEST COMPLETE")
print("=" * 55)
print(f"Phase 1 baseline : +3.22 dB")
print(f"Phase 2 best SDR : {best_val_sdr:+.2f} dB")
print()

if best_val_sdr >= 3.0:
    print(" Phase 2 model trains correctly")
    print("  Continue training to improve further")
else:
    print("Model is learning — continue training for more epochs")


Device: cuda

Building GSN Phase 2 model...

Bottleneck freq bins: 64
Harmonic graph stats:
  Nodes (freq bins) : 64
  Edges             : 158
  Avg degree        : 2.5
  Isolated nodes    : 9
GSNUNet ready | params: 2,301,634

MUSDB18Dataset  split=train  target=vocals
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=test  target=vocals
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)
Training samples   : 1000
Validation samples : 50

Loading Phase 1 weights for encoder/decoder...
  Loaded  : 98 weight tensors from Phase 1
  Skipped : 12 tensors (bottleneck is new)

PHASE 2 TRAINING — 5 epochs (smoke test)
Phase 1 baseline: +3.22 dB
Phase 2 goal    : match or exceed Phase 1

Epoch 01 | Batch 000/500 | Loss: -0.5267 | SI-SDR: +2.08 dB
Epoch 01 | Batch 050/500 | Loss: -0.0393 | SI-SDR: +1.23 dB
Epoch 01 | Batch 100/500 | Loss: -0.0646 |

KeyboardInterrupt: 

In [16]:
# ============================================================
# PHASE 2 TRAINING - FIXED VERSION
# Two learning rates: small for pretrained, large for new GCN
# ============================================================

import os
import sys
import gc
import torch

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
dataset_path = "D:/dataset"

if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
torch.manual_seed(42)

from src.models.unet import UNetConfig
from src.models.gsn_unet import GSNUNet
from src.audio.dsp import STFTConfig, STFTProcessor, AudioAugmenter
from src.training.losses import Phase1Loss, compute_si_sdr
from src.data.musdb_dataset import DataConfig, create_dataloaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print()

# -------------------------------------------------------
# Configs — same as before
# -------------------------------------------------------
stft_config = STFTConfig(
    sample_rate=44100,
    n_fft=2048,
    hop_length=512,
    win_length=2048,
)

model_config = UNetConfig(
    n_freq_bins=stft_config.n_freq_bins,
    base_channels=32,
    depth=4,
    pool_freq=True,
    use_grad_checkpoint=False,
    dropout=0.1,
)

data_config = DataConfig(
    dataset_path=dataset_path,
    sample_rate=44100,
    chunk_duration=4.0,
    batch_size=2,
    num_workers=0,
)

save_folder = os.path.join(project_root, "checkpoints", "phase2")
os.makedirs(save_folder, exist_ok=True)
checkpoint_path = os.path.join(save_folder, "best_model.pt")

max_epochs = 5
log_every = 50

# -------------------------------------------------------
# Build model
# -------------------------------------------------------
print("Building GSN Phase 2 model...")
print()

stft_processor = STFTProcessor(stft_config).to(device)

model = GSNUNet(
    config=model_config,
    sample_rate=44100,
    n_fft=2048,
    max_harmonic=5,
).to(device)

loss_function = Phase1Loss(w_l1=0.7, w_sisnr=0.3)

# -------------------------------------------------------
# TWO LEARNING RATES - this is the key fix
#
# Pretrained encoder/decoder: lr = 1e-4 (small, protect weights)
# New GCN bottleneck:         lr = 1e-3 (larger, needs to learn from scratch)
# -------------------------------------------------------

# Separate parameters by name
pretrained_params = []
new_gcn_params = []

for name, param in model.named_parameters():
    if "bottleneck" in name:
        new_gcn_params.append(param)
    else:
        pretrained_params.append(param)

print(f"Pretrained params (encoder/decoder): {sum(p.numel() for p in pretrained_params):,}")
print(f"New GCN params (bottleneck)        : {sum(p.numel() for p in new_gcn_params):,}")
print()

optimizer = torch.optim.Adam(
    [
        {"params": pretrained_params, "lr": 1e-4},   # small LR for pretrained
        {"params": new_gcn_params,    "lr": 1e-3},   # larger LR for new GCN
    ],
    weight_decay=1e-4,
)

augmenter = AudioAugmenter(
    gain_range=(0.7, 1.3),
    swap_prob=0.5,
    seed=42,
)

train_loader, val_loader = create_dataloaders(
    config=data_config,
    target_stem="vocals",
    augmenter=augmenter,
)

# -------------------------------------------------------
# Load Phase 1 weights
# -------------------------------------------------------
phase1_checkpoint = os.path.join(
    project_root,
    "checkpoints",
    "phase1",
    "phase1_final_3.22dB.pt",
)

if os.path.exists(phase1_checkpoint):
    print("Loading Phase 1 weights...")
    p1_state = torch.load(phase1_checkpoint, map_location=device)["model_state"]
    gsn_state = model.state_dict()

    loaded = 0
    skipped = 0

    for key, value in p1_state.items():
        if key in gsn_state and gsn_state[key].shape == value.shape:
            gsn_state[key] = value
            loaded += 1
        else:
            skipped += 1

    model.load_state_dict(gsn_state)
    print(f"  Loaded  : {loaded} tensors from Phase 1")
    print(f"  Skipped : {skipped} tensors (GCN bottleneck is new)")
    print()

# -------------------------------------------------------
# Verify model starts well using a quick forward pass check
# -------------------------------------------------------
print("Verifying starting SI-SDR with Phase 1 weights...")
model.eval()

with torch.no_grad():
    # Load one real batch
    sample_mixture, sample_target = next(iter(train_loader))
    sample_mixture = sample_mixture.to(device)
    sample_target  = sample_target.to(device)

    mix_mag, mix_phase, orig_len = stft_processor(sample_mixture)
    tgt_mag, _, _ = stft_processor(sample_target)

    masks = [model(mix_mag[:, ch:ch+1]) for ch in range(mix_mag.shape[1])]
    pred_mask = torch.cat(masks, dim=1)
    pred_mag  = mix_mag * pred_mask

    pred_stft  = stft_processor.magnitude_phase_to_complex(pred_mag, mix_phase)
    pred_audio = stft_processor.istft(pred_stft, length=orig_len)

    starting_sdr = compute_si_sdr(pred_audio.cpu(), sample_target.cpu())
    print(f"  Starting SI-SDR: {starting_sdr:+.2f} dB")
    print(f"  Phase 1 baseline was: +3.22 dB")

    if starting_sdr < 0:
        print()
        print("  WARNING: Starting SI-SDR is very low.")
        print("  This means the GCN bottleneck is corrupting the signal.")
        print("  The residual_scale=0 init should prevent this.")
        print("  Check if gcn_bottleneck.py was saved correctly.")
    else:
        print("  Starting point looks good. Training should improve from here.")

    del sample_mixture, sample_target, mix_mag, mix_phase
    del pred_mask, pred_mag, pred_stft, pred_audio

print()

# -------------------------------------------------------
# Helper
# -------------------------------------------------------
def cpu_si_sdr(pred: torch.Tensor, target: torch.Tensor) -> float:
    return compute_si_sdr(pred.detach().cpu(), target.detach().cpu())

# -------------------------------------------------------
# Training loop
# -------------------------------------------------------
print("=" * 55)
print(f"PHASE 2 TRAINING — {max_epochs} epochs")
print("=" * 55)
print("Encoder/decoder LR : 1e-4  (pretrained, protect)")
print("GCN bottleneck LR  : 1e-3  (new, needs to learn)")
print("Phase 1 baseline   : +3.22 dB")
print()

best_val_sdr = float("-inf")
history = {
    "train_loss": [], "val_loss": [],
    "train_si_sdr": [], "val_si_sdr": [],
}

for epoch in range(1, max_epochs + 1):

    # ---- Train ----
    model.train()
    train_loss_sum = 0.0
    train_sdr_sum  = 0.0
    train_sdr_count = 0

    for batch_idx, (mixture_audio, target_audio) in enumerate(train_loader):
        mixture_audio = mixture_audio.to(device)
        target_audio  = target_audio.to(device)

        mixture_mag, mixture_phase, original_length = stft_processor(mixture_audio)
        target_mag, _, _ = stft_processor(target_audio)

        masks = [model(mixture_mag[:, ch:ch+1]) for ch in range(mixture_mag.shape[1])]
        predicted_mask = torch.cat(masks, dim=1)
        predicted_mag  = mixture_mag * predicted_mask

        predicted_stft  = stft_processor.magnitude_phase_to_complex(predicted_mag, mixture_phase)
        predicted_audio = stft_processor.istft(predicted_stft, length=original_length)

        loss_dict  = loss_function(predicted_mag, target_mag, predicted_audio, target_audio)
        total_loss = loss_dict["loss"]

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_loss_sum += total_loss.item()

        if batch_idx % log_every == 0:
            sdr = cpu_si_sdr(predicted_audio, target_audio)
            train_sdr_sum   += sdr
            train_sdr_count += 1
            avg_loss = train_loss_sum / (batch_idx + 1)
            avg_sdr  = train_sdr_sum / train_sdr_count
            print(
                f"Epoch {epoch:02d} | Batch {batch_idx:03d}/{len(train_loader)} | "
                f"Loss: {avg_loss:.4f} | SI-SDR: {avg_sdr:+.2f} dB"
            )

        del mixture_audio, target_audio
        del mixture_mag, mixture_phase, target_mag
        del predicted_mask, predicted_mag, predicted_stft, predicted_audio
        del loss_dict, total_loss

    avg_train_loss = train_loss_sum / len(train_loader)
    avg_train_sdr  = train_sdr_sum / train_sdr_count if train_sdr_count > 0 else 0.0

    # ---- Validate ----
    model.eval()
    val_loss_sum = 0.0
    val_sdr_sum  = 0.0
    val_count    = 0

    with torch.no_grad():
        for mixture_audio, target_audio in val_loader:
            mixture_audio = mixture_audio.to(device)
            target_audio  = target_audio.to(device)

            mixture_mag, mixture_phase, original_length = stft_processor(mixture_audio)
            target_mag, _, _ = stft_processor(target_audio)

            masks = [model(mixture_mag[:, ch:ch+1]) for ch in range(mixture_mag.shape[1])]
            predicted_mask  = torch.cat(masks, dim=1)
            predicted_mag   = mixture_mag * predicted_mask
            predicted_stft  = stft_processor.magnitude_phase_to_complex(predicted_mag, mixture_phase)
            predicted_audio = stft_processor.istft(predicted_stft, length=original_length)

            loss_dict = loss_function(predicted_mag, target_mag, predicted_audio, target_audio)
            val_loss_sum += loss_dict["loss"].item()
            val_sdr_sum  += cpu_si_sdr(predicted_audio, target_audio)
            val_count    += 1

            del mixture_audio, target_audio
            del mixture_mag, mixture_phase, target_mag
            del predicted_mask, predicted_mag, predicted_stft, predicted_audio
            del loss_dict

    avg_val_loss = val_loss_sum / val_count
    avg_val_sdr  = val_sdr_sum  / val_count

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_si_sdr"].append(avg_train_sdr)
    history["val_si_sdr"].append(avg_val_sdr)

    print(f"\n{'=' * 55}")
    print(f"Epoch {epoch} done")
    print(f"  Train Loss : {avg_train_loss:.4f} | Val Loss : {avg_val_loss:.4f}")
    print(f"  Train SDR  : {avg_train_sdr:+.2f} dB | Val SDR  : {avg_val_sdr:+.2f} dB")
    print(f"  Phase 1 baseline: +3.22 dB")
    print(f"{'=' * 55}\n")

    if avg_val_sdr > best_val_sdr:
        best_val_sdr = avg_val_sdr
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss": avg_val_loss,
            "val_si_sdr": avg_val_sdr,
            "history": history,
        }, checkpoint_path)
        print(f"★ New best saved! Val SI-SDR = {avg_val_sdr:+.2f} dB\n")

# -------------------------------------------------------
# Final summary
# -------------------------------------------------------
print("=" * 55)
print("PHASE 2 SMOKE TEST DONE")
print("=" * 55)
print(f"Phase 1 baseline : +3.22 dB")
print(f"Phase 2 best SDR : {best_val_sdr:+.2f} dB")
print()

if best_val_sdr >= 3.0:
    print("✓ Phase 2 model is working correctly.")
    print("  Continue training for more epochs.")
elif best_val_sdr >= 0.0:
    print("Model is learning. Send me this output and I will diagnose.")
else:
    print("Something is still wrong. Send me the starting SI-SDR check result.")

Device: cuda

Building GSN Phase 2 model...

Bottleneck freq bins: 64
Harmonic graph stats:
  Nodes (freq bins) : 64
  Edges             : 158
  Avg degree        : 2.5
  Isolated nodes    : 9
GSNUNet ready | params: 2,301,634
Pretrained params (encoder/decoder): 2,169,025
New GCN params (bottleneck)        : 132,609


MUSDB18Dataset  split=train  target=vocals
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=test  target=vocals
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)
Loading Phase 1 weights...
  Loaded  : 98 tensors from Phase 1
  Skipped : 12 tensors (GCN bottleneck is new)

Verifying starting SI-SDR with Phase 1 weights...
  Starting SI-SDR: +4.08 dB
  Phase 1 baseline was: +3.22 dB
  Starting point looks good. Training should improve from here.

PHASE 2 TRAINING — 5 epochs
Encoder/decoder LR : 1e-4  (pretrained, protect)
G

In [17]:
# ============================================================
# PHASE 2: CONTINUE TRAINING — 20 more epochs
# Goal: exceed Phase 1 baseline of +3.22 dB
# ============================================================

import os
import sys
import gc
import torch

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
dataset_path = "D:/dataset"

if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
torch.manual_seed(42)

from src.models.unet import UNetConfig
from src.models.gsn_unet import GSNUNet
from src.audio.dsp import STFTConfig, STFTProcessor, AudioAugmenter
from src.training.losses import Phase1Loss, compute_si_sdr
from src.data.musdb_dataset import DataConfig, create_dataloaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print()

# -------------------------------------------------------
# Configs
# -------------------------------------------------------
stft_config = STFTConfig(
    sample_rate=44100,
    n_fft=2048,
    hop_length=512,
    win_length=2048,
)

model_config = UNetConfig(
    n_freq_bins=stft_config.n_freq_bins,
    base_channels=32,
    depth=4,
    pool_freq=True,
    use_grad_checkpoint=False,
    dropout=0.1,
)

data_config = DataConfig(
    dataset_path=dataset_path,
    sample_rate=44100,
    chunk_duration=4.0,
    batch_size=2,
    num_workers=0,
)

save_folder = os.path.join(project_root, "checkpoints", "phase2")
checkpoint_path = os.path.join(save_folder, "best_model.pt")
more_epochs = 20
log_every = 50

# -------------------------------------------------------
# Build model
# -------------------------------------------------------
stft_processor = STFTProcessor(stft_config).to(device)

model = GSNUNet(
    config=model_config,
    sample_rate=44100,
    n_fft=2048,
    max_harmonic=5,
).to(device)

loss_function = Phase1Loss(w_l1=0.7, w_sisnr=0.3)

# Same two learning rates that worked
pretrained_params = []
new_gcn_params = []

for name, param in model.named_parameters():
    if "bottleneck" in name:
        new_gcn_params.append(param)
    else:
        pretrained_params.append(param)

optimizer = torch.optim.Adam(
    [
        {"params": pretrained_params, "lr": 1e-4},
        {"params": new_gcn_params,    "lr": 5e-4},  # slightly lower than before
    ],
    weight_decay=1e-4,
)

augmenter = AudioAugmenter(
    gain_range=(0.7, 1.3),
    swap_prob=0.5,
    seed=42,
)

train_loader, val_loader = create_dataloaders(
    config=data_config,
    target_stem="vocals",
    augmenter=augmenter,
)

# -------------------------------------------------------
# Load Phase 2 checkpoint
# -------------------------------------------------------
if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"No Phase 2 checkpoint found at: {checkpoint_path}")

print(f"Loading Phase 2 checkpoint: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, map_location=device)

model.load_state_dict(checkpoint["model_state"])
# Do NOT load optimizer state — we are using a new LR schedule

history = checkpoint["history"]
best_val_sdr = checkpoint["val_si_sdr"]
start_epoch = checkpoint["epoch"] + 1

print(f"Resuming from epoch {start_epoch - 1}")
print(f"Best val SI-SDR so far : {best_val_sdr:+.2f} dB")
print(f"Phase 1 baseline       : +3.22 dB")
print(f"Running epochs         : {start_epoch} to {start_epoch + more_epochs - 1}")
print()

# -------------------------------------------------------
# Helper
# -------------------------------------------------------
def cpu_si_sdr(pred: torch.Tensor, target: torch.Tensor) -> float:
    return compute_si_sdr(pred.detach().cpu(), target.detach().cpu())

# -------------------------------------------------------
# Training loop
# -------------------------------------------------------
print("=" * 55)
print(f"PHASE 2 TRAINING — {more_epochs} more epochs")
print("=" * 55)
print()

for epoch in range(start_epoch, start_epoch + more_epochs):

    # ---- Train ----
    model.train()
    train_loss_sum  = 0.0
    train_sdr_sum   = 0.0
    train_sdr_count = 0

    for batch_idx, (mixture_audio, target_audio) in enumerate(train_loader):
        mixture_audio = mixture_audio.to(device)
        target_audio  = target_audio.to(device)

        mixture_mag, mixture_phase, original_length = stft_processor(mixture_audio)
        target_mag, _, _ = stft_processor(target_audio)

        masks = [model(mixture_mag[:, ch:ch+1]) for ch in range(mixture_mag.shape[1])]
        predicted_mask  = torch.cat(masks, dim=1)
        predicted_mag   = mixture_mag * predicted_mask
        predicted_stft  = stft_processor.magnitude_phase_to_complex(predicted_mag, mixture_phase)
        predicted_audio = stft_processor.istft(predicted_stft, length=original_length)

        loss_dict  = loss_function(predicted_mag, target_mag, predicted_audio, target_audio)
        total_loss = loss_dict["loss"]

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_loss_sum += total_loss.item()

        if batch_idx % log_every == 0:
            sdr = cpu_si_sdr(predicted_audio, target_audio)
            train_sdr_sum   += sdr
            train_sdr_count += 1
            avg_loss = train_loss_sum / (batch_idx + 1)
            avg_sdr  = train_sdr_sum / train_sdr_count
            print(
                f"Epoch {epoch:02d} | Batch {batch_idx:03d}/{len(train_loader)} | "
                f"Loss: {avg_loss:.4f} | SI-SDR: {avg_sdr:+.2f} dB"
            )

        del mixture_audio, target_audio
        del mixture_mag, mixture_phase, target_mag
        del predicted_mask, predicted_mag, predicted_stft, predicted_audio
        del loss_dict, total_loss

    avg_train_loss = train_loss_sum / len(train_loader)
    avg_train_sdr  = train_sdr_sum / train_sdr_count if train_sdr_count > 0 else 0.0

    # ---- Validate ----
    model.eval()
    val_loss_sum = 0.0
    val_sdr_sum  = 0.0
    val_count    = 0

    with torch.no_grad():
        for mixture_audio, target_audio in val_loader:
            mixture_audio = mixture_audio.to(device)
            target_audio  = target_audio.to(device)

            mixture_mag, mixture_phase, original_length = stft_processor(mixture_audio)
            target_mag, _, _ = stft_processor(target_audio)

            masks = [model(mixture_mag[:, ch:ch+1]) for ch in range(mixture_mag.shape[1])]
            predicted_mask  = torch.cat(masks, dim=1)
            predicted_mag   = mixture_mag * predicted_mask
            predicted_stft  = stft_processor.magnitude_phase_to_complex(predicted_mag, mixture_phase)
            predicted_audio = stft_processor.istft(predicted_stft, length=original_length)

            loss_dict = loss_function(predicted_mag, target_mag, predicted_audio, target_audio)
            val_loss_sum += loss_dict["loss"].item()
            val_sdr_sum  += cpu_si_sdr(predicted_audio, target_audio)
            val_count    += 1

            del mixture_audio, target_audio
            del mixture_mag, mixture_phase, target_mag
            del predicted_mask, predicted_mag, predicted_stft, predicted_audio
            del loss_dict

    avg_val_loss = val_loss_sum / val_count
    avg_val_sdr  = val_sdr_sum  / val_count

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_si_sdr"].append(avg_train_sdr)
    history["val_si_sdr"].append(avg_val_sdr)

    # Mark if we beat Phase 1
    beat_p1 = "✓ BEATS PHASE 1!" if avg_val_sdr > 3.22 else ""

    print(f"\n{'=' * 55}")
    print(f"Epoch {epoch} done")
    print(f"  Train Loss : {avg_train_loss:.4f} | Val Loss : {avg_val_loss:.4f}")
    print(f"  Train SDR  : {avg_train_sdr:+.2f} dB | Val SDR : {avg_val_sdr:+.2f} dB  {beat_p1}")
    print(f"  Best so far: {max(history['val_si_sdr']):+.2f} dB | Phase 1: +3.22 dB")
    print(f"{'=' * 55}\n")

    if avg_val_sdr > best_val_sdr:
        best_val_sdr = avg_val_sdr
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss": avg_val_loss,
            "val_si_sdr": avg_val_sdr,
            "history": history,
        }, checkpoint_path)
        print(f"★ New best saved! Val SI-SDR = {avg_val_sdr:+.2f} dB\n")

# -------------------------------------------------------
# Final summary
# -------------------------------------------------------
best_sdr = max(history["val_si_sdr"])

print("=" * 55)
print("PHASE 2 TRAINING RUN COMPLETE")
print("=" * 55)
print(f"Phase 1 baseline : +3.22 dB")
print(f"Phase 2 best SDR : {best_sdr:+.2f} dB")

improvement = best_sdr - 3.22
print(f"Improvement      : {improvement:+.2f} dB")
print()

if best_sdr > 3.22:
    print("Phase 2 EXCEEDS Phase 1 baseline!")
    print("  The Harmonic GCN bottleneck is helping.")
    print("  Tell me the result and we move to Phase 3: CLAP Semantic Steering.")
else:
    print("Close to Phase 1.")

Device: cuda

Bottleneck freq bins: 64
Harmonic graph stats:
  Nodes (freq bins) : 64
  Edges             : 158
  Avg degree        : 2.5
  Isolated nodes    : 9
GSNUNet ready | params: 2,301,634

MUSDB18Dataset  split=train  target=vocals
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=test  target=vocals
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)
Loading Phase 2 checkpoint: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\checkpoints\phase2\best_model.pt
Resuming from epoch 5
Best val SI-SDR so far : +3.12 dB
Phase 1 baseline       : +3.22 dB
Running epochs         : 6 to 25

PHASE 2 TRAINING — 20 more epochs

Epoch 06 | Batch 000/500 | Loss: -1.7344 | SI-SDR: +6.56 dB
Epoch 06 | Batch 050/500 | Loss: -0.5657 | SI-SDR: +6.94 dB
Epoch 06 | Batch 100/500 | Loss: -0.5283 | SI-SDR: +4.58 dB
Epoch 06 | Batch 150/500 | Loss: -0.49

In [1]:
import sys
import torch

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

from src.models.unet import MagnitudeUNet, UNetConfig
from src.models.gsn_unet import GSNUNet

config = UNetConfig(
    n_freq_bins=1025,
    base_channels=32,
    depth=4,
    pool_freq=True,
    dropout=0.1,
)

phase1 = MagnitudeUNet(config)
phase2 = GSNUNet(config, sample_rate=44100, n_fft=2048)

p1 = phase1.count_parameters()
p2 = phase2.count_parameters()

print(f"Phase 1 parameters : {p1:,}")
print(f"Phase 2 parameters : {p2:,}")
print(f"Reduction          : {(1 - p2/p1)*100:.1f}%")
print()

# Break down Phase 2 by component
print("Phase 2 breakdown:")
for name, module in phase2.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f"  {name:20s} : {params:,}")

MagnitudeUNet | channels=[32, 64, 128, 256] | pool=freq+time (2×2) | params=3,349,697
Bottleneck freq bins: 64
Harmonic graph stats:
  Nodes (freq bins) : 64
  Edges             : 158
  Avg degree        : 2.5
  Isolated nodes    : 9
GSNUNet ready | params: 2,301,634
Phase 1 parameters : 3,349,697
Phase 2 parameters : 2,301,634
Reduction          : 31.3%

Phase 2 breakdown:
  encoders             : 1,172,640
  bottleneck           : 132,609
  decoders             : 996,352
  output_conv          : 33
  sigmoid              : 0


In [2]:
# Quick check: is our GCN actually propagating information?
# This verifies the harmonic edges are working

import sys
import torch

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

from src.models.harmonic_graph import build_harmonic_edges, get_edge_stats
from src.models.gcn_bottleneck import HarmonicGCNBottleneck

# Build the graph
edge_index = build_harmonic_edges(
    n_freq_bins=64,
    sample_rate=44100,
    n_fft=128,
    max_harmonic=5,
)

stats = get_edge_stats(edge_index, 64)

print("GCN Implementation Check")
print("-" * 40)
print(f"Nodes          : {stats['num_nodes']}")
print(f"Edges          : {stats['num_edges']}")
print(f"Avg degree     : {stats['avg_degree']:.2f}")
print(f"Isolated nodes : {stats['isolated_nodes']}")
print()

# Check if information actually flows through the GCN
# Test: put a signal only on node 0, check if neighbours receive it
bottleneck = HarmonicGCNBottleneck(channels=8, edge_index=edge_index)
bottleneck.eval()

# All zeros except node 0
test_input = torch.zeros(1, 8, 64, 10)   # [B=1, C=8, freq=64, T=10]
test_input[:, :, 0, :] = 1.0             # only node 0 is active

with torch.no_grad():
    output = bottleneck(test_input)

# Because residual_scale starts near 0 after training,
# let us check the GCN output directly before residual
print("Information propagation test:")
print(f"  Input node 0 value  : {test_input[0, 0, 0, 0].item():.4f}")
print(f"  Output node 0 value : {output[0, 0, 0, 0].item():.4f}")

# Check if harmonic neighbours of node 0 received signal
# Node 0 connects to its harmonics
neighbours = edge_index[1][edge_index[0] == 0].tolist()
if neighbours:
    print(f"  Harmonic neighbours of node 0: {neighbours}")
    for nb in neighbours[:3]:
        val = output[0, 0, nb, 0].item()
        print(f"    Node {nb:3d} output: {val:.4f}")
else:
    print("  Node 0 has no harmonic neighbours (expected for DC bin)")

print()
print("If neighbour values differ from zero, information is propagating correctly.")

GCN Implementation Check
----------------------------------------
Nodes          : 64
Edges          : 158
Avg degree     : 2.47
Isolated nodes : 9

Information propagation test:
  Input node 0 value  : 1.0000
  Output node 0 value : 1.0000
  Node 0 has no harmonic neighbours (expected for DC bin)

If neighbour values differ from zero, information is propagating correctly.


In [3]:
# Save some audio samples so you can actually listen to the separation quality

import os
import sys
import torch
import torchaudio

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
dataset_path = "D:/dataset"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

from src.models.unet import UNetConfig
from src.models.gsn_unet import GSNUNet
from src.audio.dsp import STFTConfig, STFTProcessor
from src.data.musdb_dataset import DataConfig, MUSDB18Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Phase 2 model
stft_config = STFTConfig()
model_config = UNetConfig(
    n_freq_bins=stft_config.n_freq_bins,
    base_channels=32,
    depth=4,
    pool_freq=True,
    dropout=0.1,
)

stft_processor = STFTProcessor(stft_config).to(device)
model = GSNUNet(model_config, sample_rate=44100, n_fft=2048).to(device)

checkpoint = torch.load(
    os.path.join(project_root, "checkpoints", "phase2", "phase2_final_3.12dB.pt"),
    map_location=device,
)
model.load_state_dict(checkpoint["model_state"])
model.eval()

print("Model loaded.")

# Load a 10 second test clip
data_config = DataConfig(
    dataset_path=dataset_path,
    sample_rate=44100,
    chunk_duration=10.0,
    batch_size=1,
    num_workers=0,
)

test_dataset = MUSDB18Dataset(
    config=data_config,
    split="test",
    target_stem="vocals",
)

# Pick 3 different songs to compare
output_dir = os.path.join(project_root, "outputs", "audio", "phase2_samples")
os.makedirs(output_dir, exist_ok=True)

print(f"Saving audio samples to: {output_dir}")
print()

def normalize_audio(audio: torch.Tensor) -> torch.Tensor:
    max_val = audio.abs().max()
    if max_val > 0:
        return audio / max_val * 0.9
    return audio

for sample_idx in [0, 5, 10]:
    mixture_audio, vocals_gt = test_dataset[sample_idx]
    song_name = test_dataset.songs[sample_idx // test_dataset.chunks_per_song]["name"]
    # Shorten name for filename
    safe_name = song_name[:30].replace(" ", "_").replace("/", "_")

    mixture_batch = mixture_audio.unsqueeze(0).to(device)
    vocals_batch  = vocals_gt.unsqueeze(0).to(device)

    with torch.no_grad():
        mix_mag, mix_phase, orig_len = stft_processor(mixture_batch)

        masks = [model(mix_mag[:, ch:ch+1]) for ch in range(mix_mag.shape[1])]
        predicted_mask = torch.cat(masks, dim=1)
        predicted_mag  = mix_mag * predicted_mask

        predicted_stft  = stft_processor.magnitude_phase_to_complex(predicted_mag, mix_phase)
        predicted_vocals = stft_processor.istft(predicted_stft, length=orig_len)

        # Also compute estimated accompaniment (mixture minus predicted vocals)
        predicted_accompaniment = mixture_batch - predicted_vocals

    # Save files
    torchaudio.save(
        os.path.join(output_dir, f"{safe_name}_1_mixture.wav"),
        normalize_audio(mixture_audio),
        44100,
    )
    torchaudio.save(
        os.path.join(output_dir, f"{safe_name}_2_predicted_vocals.wav"),
        normalize_audio(predicted_vocals.squeeze(0).cpu()),
        44100,
    )
    torchaudio.save(
        os.path.join(output_dir, f"{safe_name}_3_ground_truth_vocals.wav"),
        normalize_audio(vocals_gt),
        44100,
    )
    torchaudio.save(
        os.path.join(output_dir, f"{safe_name}_4_accompaniment.wav"),
        normalize_audio(predicted_accompaniment.squeeze(0).cpu()),
        44100,
    )

    from src.training.losses import compute_si_sdr
    si_sdr = compute_si_sdr(predicted_vocals.cpu(), vocals_batch.cpu())

    print(f"Song: {song_name[:50]}")
    print(f"  SI-SDR: {si_sdr:+.2f} dB")
    print(f"  Saved 4 audio files")
    print()

print("Done. Listen to the files and compare:")
print("  _1_mixture          = input (what you feed the model)")
print("  _2_predicted_vocals = what our model extracted")
print("  _3_ground_truth     = what perfect separation sounds like")
print("  _4_accompaniment    = everything minus predicted vocals")
print()
print("At +3 dB, expect:")
print("  - vocals are audible and somewhat isolated")
print("  - some bleed from other instruments")
print("  - slightly muffled/watery quality")
print("  - this is NORMAL for magnitude-only separation")
print()
print("Phase 4 (complex masking) will fix the muffled quality.")

Bottleneck freq bins: 64
Harmonic graph stats:
  Nodes (freq bins) : 64
  Edges             : 158
  Avg degree        : 2.5
  Isolated nodes    : 9
GSNUNet ready | params: 2,301,634


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\\checkpoints\\phase2\\phase2_final_3.12dB.pt'

In [4]:
import os
import torch

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"

print("CHECKPOINT FOLDER CONTENTS")
print("=" * 55)

checkpoint_root = os.path.join(project_root, "checkpoints")

for phase_folder in sorted(os.listdir(checkpoint_root)):
    phase_path = os.path.join(checkpoint_root, phase_folder)
    
    if not os.path.isdir(phase_path):
        continue
    
    files = os.listdir(phase_path)
    print(f"\n{phase_folder}/")
    
    if len(files) == 0:
        print("  (empty)")
        continue
    
    for fname in sorted(files):
        fpath = os.path.join(phase_path, fname)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {fname}  ({size_mb:.1f} MB)")
        
        # Try to open .pt files and show what's inside
        if fname.endswith(".pt"):
            try:
                ckpt = torch.load(fpath, map_location="cpu")
                if isinstance(ckpt, dict):
                    print(f"    Keys: {list(ckpt.keys())}")
                    if "epoch" in ckpt:
                        print(f"    Epoch: {ckpt['epoch']}")
                    if "val_si_sdr" in ckpt:
                        print(f"    Val SI-SDR: {ckpt['val_si_sdr']:+.2f} dB")
                    if "val_loss" in ckpt:
                        print(f"    Val Loss: {ckpt['val_loss']:.4f}")
            except Exception as e:
                print(f"    Could not read: {e}")

CHECKPOINT FOLDER CONTENTS

phase1/
  best_model.pt  (40.3 MB)
    Keys: ['epoch', 'model_state', 'optimizer_state', 'val_loss', 'val_si_sdr', 'history']
    Epoch: 33
    Val SI-SDR: +3.22 dB
    Val Loss: -0.7756
  phase1_final_3.22dB.pt  (40.3 MB)
    Keys: ['epoch', 'model_state', 'optimizer_state', 'val_loss', 'val_si_sdr', 'history']
    Epoch: 33
    Val SI-SDR: +3.22 dB
    Val Loss: -0.7756

phase2/
  best_model.pt  (27.7 MB)
    Keys: ['epoch', 'model_state', 'optimizer_state', 'val_loss', 'val_si_sdr', 'history']
    Epoch: 5
    Val SI-SDR: +3.12 dB
    Val Loss: -0.7235


In [5]:
import os
import shutil
import torch

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"

src = os.path.join(project_root, "checkpoints", "phase2", "best_model.pt")
dst = os.path.join(project_root, "checkpoints", "phase2", "phase2_final_3.12dB.pt")

shutil.copy(src, dst)

# Verify
ckpt = torch.load(dst, map_location="cpu")
print("Phase 2 final checkpoint saved:")
print(f"  File    : {dst}")
print(f"  Epoch   : {ckpt['epoch']}")
print(f"  SI-SDR  : {ckpt['val_si_sdr']:+.2f} dB")
print(f"  Size    : {os.path.getsize(dst)/1e6:.1f} MB")
print()
print("Summary of all checkpoints:")
print(f"  Phase 1 : +3.22 dB  (40.3 MB)  ✓")
print(f"  Phase 2 : +3.12 dB  (27.7 MB)  ✓")
print()
print("Both phases complete. Ready for Phase 3.")

Phase 2 final checkpoint saved:
  File    : C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\checkpoints\phase2\phase2_final_3.12dB.pt
  Epoch   : 5
  SI-SDR  : +3.12 dB
  Size    : 27.7 MB

Summary of all checkpoints:
  Phase 1 : +3.22 dB  (40.3 MB)  ✓
  Phase 2 : +3.12 dB  (27.7 MB)  ✓

Both phases complete. Ready for Phase 3.


In [1]:
# Generate listenable audio samples from your Phase 2 model
# Output: .wav files you can double-click and play

import os
import sys
import torch
import torchaudio

project_root = "C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION"
dataset_path = "D:/dataset"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

for key in list(sys.modules.keys()):
    if "src" in key:
        del sys.modules[key]

from src.models.unet import UNetConfig
from src.models.gsn_unet import GSNUNet
from src.audio.dsp import STFTConfig, STFTProcessor
from src.data.musdb_dataset import DataConfig, MUSDB18Dataset
from src.training.losses import compute_si_sdr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
stft_config = STFTConfig()
model_config = UNetConfig(
    n_freq_bins=stft_config.n_freq_bins,
    base_channels=32,
    depth=4,
    pool_freq=True,
    dropout=0.1,
)

stft_processor = STFTProcessor(stft_config).to(device)
model = GSNUNet(model_config, sample_rate=44100, n_fft=2048).to(device)

checkpoint = torch.load(
    os.path.join(project_root, "checkpoints", "phase2", "phase2_final_3.12dB.pt"),
    map_location=device,
)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print("✓ Model loaded")

# Load test songs (10 seconds each)
data_config = DataConfig(
    dataset_path=dataset_path,
    sample_rate=44100,
    chunk_duration=10.0,
    batch_size=1,
    num_workers=0,
)

test_dataset = MUSDB18Dataset(
    config=data_config,
    split="test",
    target_stem="vocals",
)

# Output folder — these are the files you will listen to
output_dir = os.path.join(project_root, "outputs", "audio", "listen_to_these")
os.makedirs(output_dir, exist_ok=True)

def normalize(audio: torch.Tensor) -> torch.Tensor:
    max_val = audio.abs().max()
    if max_val > 0:
        return audio / max_val * 0.9
    return audio

print(f"\nSaving audio to: {output_dir}")
print("=" * 55)

# Generate 3 examples
for i, song_idx in enumerate([0, 3, 7]):
    mixture_audio, vocals_gt = test_dataset[song_idx]
    song_name = test_dataset.songs[song_idx]["name"]
    safe_name = f"song{i+1}_{song_name[:25].replace(' ', '_')}"

    mixture_batch = mixture_audio.unsqueeze(0).to(device)

    with torch.no_grad():
        mix_mag, mix_phase, orig_len = stft_processor(mixture_batch)

        masks = [model(mix_mag[:, ch:ch+1]) for ch in range(mix_mag.shape[1])]
        pred_mask  = torch.cat(masks, dim=1)
        pred_mag   = mix_mag * pred_mask
        pred_stft  = stft_processor.magnitude_phase_to_complex(pred_mag, mix_phase)
        pred_vocals = stft_processor.istft(pred_stft, length=orig_len)
        pred_accompaniment = mixture_batch - pred_vocals

    si_sdr = compute_si_sdr(pred_vocals.cpu(), mixture_audio.unsqueeze(0))

    # Save the 4 wav files
    files = {
        f"{safe_name}__1_MIXTURE.wav":          normalize(mixture_audio),
        f"{safe_name}__2_PREDICTED_VOCALS.wav":  normalize(pred_vocals.squeeze(0).cpu()),
        f"{safe_name}__3_GROUND_TRUTH.wav":      normalize(vocals_gt),
        f"{safe_name}__4_ACCOMPANIMENT.wav":     normalize(pred_accompaniment.squeeze(0).cpu()),
    }

    for filename, audio in files.items():
        torchaudio.save(
            os.path.join(output_dir, filename),
            audio.float(),
            44100,
        )

    print(f"\nSong {i+1}: {song_name}")
    print(f"  SI-SDR  : {si_sdr:+.2f} dB")
    print(f"  Files saved:")
    for filename in files:
        print(f"    {filename}")

print()
print("=" * 55)
print("DONE. Open this folder in Windows Explorer:")
print(f"  {output_dir}")
print()
print("Listen in this order for each song:")
print("  __1_MIXTURE          = the original full song (input)")
print("  __2_PREDICTED_VOCALS = what our AI extracted")
print("  __3_GROUND_TRUTH     = perfect separation (target)")
print("  __4_ACCOMPANIMENT    = drums + bass + other (mixture minus vocals)")
print()
print("What to expect at +3 dB:")
print("  ✓ Vocals are clearly present and somewhat isolated")
print("  ✓ Other instruments are reduced but not gone")
print("  ✗ Slight muffled/watery quality (fixed in Phase 4)")
print("  ✗ Some bleed from other instruments (normal at this stage)")

Bottleneck freq bins: 64
Harmonic graph stats:
  Nodes (freq bins) : 64
  Edges             : 158
  Avg degree        : 2.5
  Isolated nodes    : 9
GSNUNet ready | params: 2,301,634
✓ Model loaded

MUSDB18Dataset  split=test  target=vocals
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=10.0s (441000 samples)

Saving audio to: C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION\outputs\audio\listen_to_these

Song 1: AM Contra - Heart Peripheral
  SI-SDR  : -3.50 dB
  Files saved:
    song1_AM_Contra_-_Heart_Periphe__1_MIXTURE.wav
    song1_AM_Contra_-_Heart_Periphe__2_PREDICTED_VOCALS.wav
    song1_AM_Contra_-_Heart_Periphe__3_GROUND_TRUTH.wav
    song1_AM_Contra_-_Heart_Periphe__4_ACCOMPANIMENT.wav

Song 2: Arise - Run Run Run
  SI-SDR  : -0.27 dB
  Files saved:
    song2_Arise_-_Run_Run_Run__1_MIXTURE.wav
    song2_Arise_-_Run_Run_Run__2_PREDICTED_VOCALS.wav
    song2_Arise_-_Run_Run_Run__3_GROUND_TRUTH.wav
    song2_Arise_-_Run_Run_Run__4_ACCOM